In [5]:
for name, df in datasets.items():

    missing = (
        df.isna()
        .sum()
        .reset_index()
    )

    missing.columns = ["column", "missing_count"]

    missing["missing_percentage"] = (
        missing["missing_count"] / len(df) * 100
    )

    missing = missing[missing["missing_count"] > 0]

    if not missing.empty:
        print(f"\n{name}")
        display(missing)


orders


,column,missing_count,missing_percentage
4,order_approved_at,160,0.160899
5,order_delivered_carrier_date,1783,1.793023
6,order_delivered_customer_date,2965,2.981668



reviews


,column,missing_count,missing_percentage
3,review_comment_title,87656,88.341530
4,review_comment_message,58247,58.702532



products


,column,missing_count,missing_percentage
1,product_category_name,610,1.851234
2,product_name_lenght,610,1.851234
3,product_description_lenght,610,1.851234
4,product_photos_qty,610,1.851234
5,product_weight_g,2,0.006070
6,product_length_cm,2,0.006070
7,product_height_cm,2,0.006070
8,product_width_cm,2,0.006070


In [6]:
primary_keys = {
    "orders": ["order_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "reviews": ["review_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"]
}

for table, keys in primary_keys.items():

    df = datasets[table]

    duplicate_keys = df.duplicated(
        subset=keys
    ).sum()

    print(
        f"{table}: "
        f"{duplicate_keys} duplicate primary-key combinations"
    )
    

orders: 0 duplicate primary-key combinations
customers: 0 duplicate primary-key combinations
products: 0 duplicate primary-key combinations
sellers: 0 duplicate primary-key combinations
reviews: 814 duplicate primary-key combinations
order_items: 0 duplicate primary-key combinations
payments: 0 duplicate primary-key combinations


In [7]:
print(
    "orders → customers:",
    orders["customer_id"].isin(
        customers["customer_id"]
    ).all()
)

print(
    "order_items → orders:",
    order_items["order_id"].isin(
        orders["order_id"]
    ).all()
)

print(
    "order_items → products:",
    order_items["product_id"].isin(
        products["product_id"]
    ).all()
)

print(
    "order_items → sellers:",
    order_items["seller_id"].isin(
        sellers["seller_id"]
    ).all()
)

print(
    "payments → orders:",
    payments["order_id"].isin(
        orders["order_id"]
    ).all()
)

print(
    "reviews → orders:",
    reviews["order_id"].isin(
        orders["order_id"]
    ).all()
)

orders → customers: True
order_items → orders: True
order_items → products: True
order_items → sellers: True
payments → orders: True
reviews → orders: True


In [8]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [9]:
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

## Data Quality Summary

The Olist dataset contains approximately 100K orders across
multiple normalized CSV files.

Key findings:

- No complete duplicate rows were identified.
- Primary-key candidates are unique.
- All tested foreign-key relationships are valid.
- Missing values are primarily present in delivery timestamps,
  product attributes, and optional review comments.
- Missing delivery timestamps are expected for orders that were
  not completed/delivered.
- Order items and payments use composite keys.
- Some orders have multiple review records, so review data
  requires aggregation before joining to order-level analysis.
- No negative price, freight, or payment values were identified.

The raw data is suitable for loading into PostgreSQL, with
appropriate handling of missing values and data types during ETL.

In [4]:
datasets = {
    "orders": orders,
    "customers": customers,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

summary = []

for name, df in datasets.items():
    summary.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum()
    })

summary_df = pd.DataFrame(summary)

summary_df


,table,rows,columns,duplicate_rows,missing_values
0,orders,99441,8,0,4908
1,customers,99441,5,0,0
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,0,145903
5,products,32951,9,0,2448
6,sellers,3095,4,0,0
7,category_translation,73,2,0,0


In [3]:
orders = pd.read_csv(DATA_PATH / "olist_orders_dataset.csv")
customers = pd.read_csv(DATA_PATH / "olist_customers_dataset.csv")
order_items = pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_PATH / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_PATH / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(
    DATA_PATH / "product_category_name_translation.csv"
)

In [2]:
DATA_PATH = Path("../data/raw")


In [1]:
import pandas as pd
from pathlib import Path